# Querying the ZEDProfiler pilot warehouse

Every completed run writes a `warehouse/` directory (plain namespaced parquet datasets, no catalog) plus a small `warehouse.duckdb` file inside it holding one `VIEW` per table -- `profiles.nuclei_profiles`, `profiles.cell_profiles`, `profiles.cytoplasm_profiles`, `profiles.organoid_profiles`, `images.image_assets`. A view is a stored query over `read_parquet(glob)`, not a copy, so it's always in sync with whatever parquet files are currently on disk.

**Views are defined with paths relative to `warehouse/`**, so this notebook needs its working directory set to that folder before connecting -- see `scripts/build_duckdb_views.py`'s docstring for why. Edit `WAREHOUSE_DIR` below to point at whichever run you want to inspect, e.g.
`/pl/active/koala/nf1-3d-pilot-workflow-db/results/<run_id>/warehouse` on Alpine.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 12)

WAREHOUSE_DIR = Path("/pl/active/koala/nf1-3d-pilot-workflow-db/results/nf0055-nf0014-warehouse-dir-retry-20260814T030514Z/warehouse")

os.chdir(WAREHOUSE_DIR)
conn = duckdb.connect("warehouse.duckdb")
conn.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,warehouse,images,image_assets,"[Metadata_Biology_PatientTumor, Metadata_Biolo...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
1,warehouse,joined,images_nuclei_cell_cytoplasm,"[Metadata_Biology_PatientTumor, Metadata_Biolo...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
2,warehouse,joined,images_organoid,"[Metadata_Biology_PatientTumor, Metadata_Biolo...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
3,warehouse,joined,nuclei_cell_cytoplasm_images,"[Metadata_Biology_PatientTumor, Metadata_Biolo...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
4,warehouse,joined,organoid_images,"[Metadata_Biology_PatientTumor, Metadata_Biolo...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
5,warehouse,profiles,cell_profiles,"[Metadata_Compartment, Metadata_Segmentation_P...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
6,warehouse,profiles,cytoplasm_profiles,"[Metadata_Compartment, Metadata_Segmentation_P...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
7,warehouse,profiles,nuclei_profiles,"[Metadata_Compartment, Metadata_Segmentation_P...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
8,warehouse,profiles,organoid_profiles,"[Metadata_Compartment, Metadata_Segmentation_P...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False


## Profile tables

One row per segmented object per image set. `head()` on each -- the metadata columns (patient, well, image ID, compartment, object ID) come first, followed by hundreds of feature columns per compartment.

In [2]:
profile_tables = ["nuclei_profiles", "cell_profiles", "cytoplasm_profiles", "organoid_profiles"]

for table in profile_tables:
    df = conn.execute(f"SELECT * FROM profiles.{table}").fetchdf()
    print(f"profiles.{table}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.iloc[:, :10].head())

profiles.nuclei_profiles: 56 rows x 903 columns


,Metadata_Compartment,Metadata_Segmentation_PrimaryChannel,Metadata_Segmentation_PrimaryChannelCode,Metadata_Segmentation_SeedChannel,Metadata_Segmentation_SeedChannelCode,Metadata_Segmentation_Method,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID
0,Nuclei,DNA,405,,,segmented_from_dna,NF0014_T1,NF0014,NF0014_T1,C4
1,Nuclei,DNA,405,,,segmented_from_dna,NF0014_T1,NF0014,NF0014_T1,C4
2,Nuclei,DNA,405,,,segmented_from_dna,NF0014_T1,NF0014,NF0014_T1,C4
3,Nuclei,DNA,405,,,segmented_from_dna,NF0014_T1,NF0014,NF0014_T1,C4
4,Nuclei,DNA,405,,,segmented_from_dna,NF0014_T1,NF0014,NF0014_T1,C4


profiles.cell_profiles: 51 rows x 903 columns


,Metadata_Compartment,Metadata_Segmentation_PrimaryChannel,Metadata_Segmentation_PrimaryChannelCode,Metadata_Segmentation_SeedChannel,Metadata_Segmentation_SeedChannelCode,Metadata_Segmentation_Method,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID
0,Cell,AGP,555,DNA,405,agp_watershed_seeded_by_nuclei,NF0014_T1,NF0014,NF0014_T1,C4
1,Cell,AGP,555,DNA,405,agp_watershed_seeded_by_nuclei,NF0014_T1,NF0014,NF0014_T1,C4
2,Cell,AGP,555,DNA,405,agp_watershed_seeded_by_nuclei,NF0014_T1,NF0014,NF0014_T1,C4
3,Cell,AGP,555,DNA,405,agp_watershed_seeded_by_nuclei,NF0014_T1,NF0014,NF0014_T1,C4
4,Cell,AGP,555,DNA,405,agp_watershed_seeded_by_nuclei,NF0014_T1,NF0014,NF0014_T1,C4


profiles.cytoplasm_profiles: 51 rows x 903 columns


,Metadata_Compartment,Metadata_Segmentation_PrimaryChannel,Metadata_Segmentation_PrimaryChannelCode,Metadata_Segmentation_SeedChannel,Metadata_Segmentation_SeedChannelCode,Metadata_Segmentation_Method,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID
0,Cytoplasm,AGP,555,DNA,405,cell_mask_minus_nuclei_mask,NF0014_T1,NF0014,NF0014_T1,C4
1,Cytoplasm,AGP,555,DNA,405,cell_mask_minus_nuclei_mask,NF0014_T1,NF0014,NF0014_T1,C4
2,Cytoplasm,AGP,555,DNA,405,cell_mask_minus_nuclei_mask,NF0014_T1,NF0014,NF0014_T1,C4
3,Cytoplasm,AGP,555,DNA,405,cell_mask_minus_nuclei_mask,NF0014_T1,NF0014,NF0014_T1,C4
4,Cytoplasm,AGP,555,DNA,405,cell_mask_minus_nuclei_mask,NF0014_T1,NF0014,NF0014_T1,C4


profiles.organoid_profiles: 3 rows x 903 columns


,Metadata_Compartment,Metadata_Segmentation_PrimaryChannel,Metadata_Segmentation_PrimaryChannelCode,Metadata_Segmentation_SeedChannel,Metadata_Segmentation_SeedChannelCode,Metadata_Segmentation_Method,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID
0,Organoid,AGP,555,,,segmented_from_agp,NF0014_T1,NF0014,NF0014_T1,C4
1,Organoid,AGP,555,,,segmented_from_agp,NF0055_T1,NF0055,NF0055_T1,B10
2,Organoid,AGP,555,,,segmented_from_agp,NF0055_T1,NF0055,NF0055_T1,B10


## Image assets

One row per raw channel image and per segmentation mask, per image set -- 8 rows per image set (4 channels + 4 compartment masks).

In [3]:
assets = conn.execute("SELECT * FROM images.image_assets").fetchdf()
print(f"images.image_assets: {assets.shape[0]} rows x {assets.shape[1]} columns")
assets.head(10)

images.image_assets: 16 rows x 19 columns


,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,...,Metadata_ImageAsset_DType,Metadata_ImageAsset_SizeZ,Metadata_ImageAsset_SizeY,Metadata_ImageAsset_SizeX,Metadata_Run_RunID,Metadata_Run_GitCommit
0,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
1,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
2,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
3,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
4,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
5,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
6,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
7,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,...,uint16,33,1537,1540,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
8,NF0055_T1,NF0055,NF0055_T1,B10,1,NF0055_T1__NF0055_T1__B10__F1,...,uint16,105,1527,1528,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db
9,NF0055_T1,NF0055,NF0055_T1,B10,1,NF0055_T1__NF0055_T1__B10__F1,...,uint16,105,1527,1528,nf0055-nf0014-warehouse-dir-retry-20260814T030...,a45e9db


## Cross-table query: joining a profile table with image assets

Views behave like any other DuckDB table, so ordinary SQL joins work across them -- here, pulling each nuclei object's row alongside metadata about its image set's raw channels and masks.

In [4]:
joined = conn.execute(
    """
    SELECT
        p.Metadata_Imaging_ImageID,
        p.Metadata_Object_ObjectID,
        a.Metadata_ImageAsset_AssetType,
        a.Metadata_ImageAsset_Channel,
        a.Metadata_ImageAsset_SizeZ,
        a.Metadata_ImageAsset_SizeY,
        a.Metadata_ImageAsset_SizeX
    FROM profiles.nuclei_profiles p
    JOIN images.image_assets a USING (Metadata_Imaging_ImageID)
    ORDER BY p.Metadata_Imaging_ImageID, p.Metadata_Object_ObjectID, a.Metadata_ImageAsset_AssetType
    """
).fetchdf()
print(f"{joined.shape[0]} rows (nuclei objects x 8 assets per image set)")
joined.head(10)

448 rows (nuclei objects x 8 assets per image set)


,Metadata_Imaging_ImageID,Metadata_Object_ObjectID,Metadata_ImageAsset_AssetType,Metadata_ImageAsset_Channel,Metadata_ImageAsset_SizeZ,Metadata_ImageAsset_SizeY,Metadata_ImageAsset_SizeX
0,NF0014_T1__NF0014_T1__C4__F2,1,raw_image,Mito,33,1537,1540
1,NF0014_T1__NF0014_T1__C4__F2,1,raw_image,AGP,33,1537,1540
2,NF0014_T1__NF0014_T1__C4__F2,1,raw_image,ER,33,1537,1540
3,NF0014_T1__NF0014_T1__C4__F2,1,raw_image,DNA,33,1537,1540
4,NF0014_T1__NF0014_T1__C4__F2,1,segmentation_mask,AGP,33,1537,1540
5,NF0014_T1__NF0014_T1__C4__F2,1,segmentation_mask,AGP,33,1537,1540
6,NF0014_T1__NF0014_T1__C4__F2,1,segmentation_mask,AGP,33,1537,1540
7,NF0014_T1__NF0014_T1__C4__F2,1,segmentation_mask,DNA,33,1537,1540
8,NF0014_T1__NF0014_T1__C4__F2,2,raw_image,Mito,33,1537,1540
9,NF0014_T1__NF0014_T1__C4__F2,2,raw_image,AGP,33,1537,1540


## Aggregate example: object counts per compartment per image set

A `UNION ALL` across the four profile views, then a simple `GROUP BY` -- the kind of summary query that's normally the reason to reach for a catalog/warehouse layer in the first place, here running directly against the parquet files with no data ever copied into DuckDB itself.

In [5]:
summary = conn.execute(
    """
    SELECT Metadata_Imaging_ImageID, Metadata_Compartment, count(*) AS object_count
    FROM (
        SELECT Metadata_Imaging_ImageID, Metadata_Compartment FROM profiles.nuclei_profiles
        UNION ALL
        SELECT Metadata_Imaging_ImageID, Metadata_Compartment FROM profiles.cell_profiles
        UNION ALL
        SELECT Metadata_Imaging_ImageID, Metadata_Compartment FROM profiles.cytoplasm_profiles
        UNION ALL
        SELECT Metadata_Imaging_ImageID, Metadata_Compartment FROM profiles.organoid_profiles
    )
    GROUP BY 1, 2
    ORDER BY 1, 2
    """
).fetchdf()
summary

,Metadata_Imaging_ImageID,Metadata_Compartment,object_count
0,NF0014_T1__NF0014_T1__C4__F2,Cell,42
1,NF0014_T1__NF0014_T1__C4__F2,Cytoplasm,42
2,NF0014_T1__NF0014_T1__C4__F2,Nuclei,45
3,NF0014_T1__NF0014_T1__C4__F2,Organoid,1
4,NF0055_T1__NF0055_T1__B10__F1,Cell,9
5,NF0055_T1__NF0055_T1__B10__F1,Cytoplasm,9
6,NF0055_T1__NF0055_T1__B10__F1,Nuclei,11
7,NF0055_T1__NF0055_T1__B10__F1,Organoid,2


## Joined views: single-cell compartments and organoid, each with image assets

Two composite views live under a separate `joined` schema, both with image-asset columns first:

- `joined.images_nuclei_cell_cytoplasm` -- Nuclei, Cell, and Cytoplasm joined together first (on `Metadata_Imaging_ImageID` + `Metadata_Object_ObjectID`, since Cell/Cytoplasm are seeded from Nuclei's segmentation and share its object IDs), then to `images.image_assets`.
- `joined.images_organoid` -- Organoid joined directly to `images.image_assets`. Organoid is segmented independently (no shared object-ID space with the single-cell compartments), so it isn't part of the first view.

Columns identical across every table (patient/plate/well/field/image IDs) are kept once, from `image_assets`. Columns that differ per compartment (`Metadata_Compartment`, the `Segmentation_*` ones) are renamed with a per-compartment prefix -- `Metadata_Nuclei_Compartment`, `Metadata_Cell_Compartment`, etc. -- so no column name collides.

In [6]:
single_cell = conn.execute("SELECT * FROM joined.images_nuclei_cell_cytoplasm").fetchdf()
print(f"joined.images_nuclei_cell_cytoplasm: {single_cell.shape[0]} rows x {single_cell.shape[1]} columns")
print("duplicate column names:", single_cell.columns.duplicated().any())
display(single_cell.iloc[:, :12].head())

joined.images_nuclei_cell_cytoplasm: 408 rows x 2705 columns
duplicate column names: False


,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,Metadata_ImageAsset_AssetID,Metadata_ImageAsset_AssetType,Metadata_ImageAsset_Channel,Metadata_ImageAsset_ChannelCode,Metadata_ImageAsset_Compartment,Metadata_ImageAsset_SegmentationMethod
0,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Organoid_mask,segmentation_mask,AGP,555,Organoid,segmented_from_agp
1,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Organoid_mask,segmentation_mask,AGP,555,Organoid,segmented_from_agp
2,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Organoid_mask,segmentation_mask,AGP,555,Organoid,segmented_from_agp
3,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Organoid_mask,segmentation_mask,AGP,555,Organoid,segmented_from_agp
4,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Organoid_mask,segmentation_mask,AGP,555,Organoid,segmented_from_agp


In [7]:
organoid = conn.execute("SELECT * FROM joined.images_organoid").fetchdf()
print(f"joined.images_organoid: {organoid.shape[0]} rows x {organoid.shape[1]} columns")
print("duplicate column names:", organoid.columns.duplicated().any())
organoid.iloc[:, :12].head()

joined.images_organoid: 24 rows x 915 columns
duplicate column names: False


,Metadata_Biology_PatientTumor,Metadata_Biology_PatientID,Metadata_Experiment_PlateID,Metadata_Experiment_WellID,Metadata_Imaging_FieldID,Metadata_Imaging_ImageID,Metadata_ImageAsset_AssetID,Metadata_ImageAsset_AssetType,Metadata_ImageAsset_Channel,Metadata_ImageAsset_ChannelCode,Metadata_ImageAsset_Compartment,Metadata_ImageAsset_SegmentationMethod
0,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Organoid_mask,segmentation_mask,AGP,555,Organoid,segmented_from_agp
1,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Cytoplasm_mask,segmentation_mask,AGP,555,Cytoplasm,cell_mask_minus_nuclei_mask
2,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Cell_mask,segmentation_mask,AGP,555,Cell,agp_watershed_seeded_by_nuclei
3,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Nuclei_mask,segmentation_mask,DNA,405,Nuclei,segmented_from_dna
4,NF0014_T1,NF0014,NF0014_T1,C4,2,NF0014_T1__NF0014_T1__C4__F2,NF0014_T1__NF0014_T1__C4__F2::Mito,raw_image,Mito,640,,


In [8]:
# Per-compartment columns stay distinguishable after the join -- no collision.
conn.execute(
    """
    SELECT DISTINCT
        Metadata_Nuclei_Compartment, Metadata_Cell_Compartment, Metadata_Cytoplasm_Compartment,
        Metadata_Nuclei_Segmentation_SeedChannel, Metadata_Cell_Segmentation_SeedChannel
    FROM joined.images_nuclei_cell_cytoplasm
    """
).fetchdf()

,Metadata_Nuclei_Compartment,Metadata_Cell_Compartment,Metadata_Cytoplasm_Compartment,Metadata_Nuclei_Segmentation_SeedChannel,Metadata_Cell_Segmentation_SeedChannel
0,Nuclei,Cell,Cytoplasm,,DNA
